In [14]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
import sys, os
from autograd import grad, hessian
import autograd.numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd
from utils import *
from sklearn.metrics import mean_squared_error
import matplotlib.colors as mcolors
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multioutput import MultiOutputRegressor



In [77]:
# Load the data 
data_dir = '../../data_n10k_splot22f.csv'
data_loaded  = np.loadtxt(data_dir, delimiter=',', dtype=float, skiprows = 1)

# 0:Met[Zsun],1:Age[Gyr],2:Rp[Rsun],3:Vinf[km/s],4:Mass1_i[MSUN],5:Mass2_i[Msun],6:R1[Rsun],7:R2[Rsun],8:Label,9:Mass1_f[MSUN],10:Mass2_f[MSUN],11:Sigma
data = data_loaded[:, [1, 2, 3, 4, 5]]
initial_masses = data_loaded[:, [4, 5]]
final_masses = data_loaded[:, [9, 10]]

# Re-assing final masses for merger cases to be all in mass1
merger_mask = data_loaded[:, 8] == 1
swap_rows = merger_mask & (final_masses[:, 0] == 0) & (final_masses[:, 1] > 0)
final_masses[swap_rows, 0], final_masses[swap_rows, 1] = (final_masses[swap_rows, 1], final_masses[swap_rows, 0])

# Creating the regression dataset as [ M1,f / Mtot,i; M2,f / Mtot,i; Mejec,f / Mtot,i ]
y_data_reg = np.array([final_masses[:,0] / (initial_masses[:,0] + initial_masses[:,1]) , final_masses[:,1] / (initial_masses[:,0] + initial_masses[:,1]), ((initial_masses[:,0] + initial_masses[:,1]) - (final_masses[:,0] + final_masses[:,1]))/ (initial_masses[:,0] + initial_masses[:,1])] ).T

# Feature transform
x_data = data
x_data[:, 0] = np.log10(x_data[:, 0]) # log10 ages
x_data[:, 2] = np.log10(x_data[:, 2]) # log10 vinfs
x_data[:, 3] = np.log(x_data[:, 3])  #ln M1,i
x_data[:, 4] = np.log(x_data[:, 4])  #ln M2,i

# Split data into 70% training, 15% validation, and 15% testing
random_state = 42
x_data_og = x_data

#Split into Training + Temporary (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(x_data, y_data_reg, test_size=0.30, random_state=42)

# Split the temporary into validation and testing sets 
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(np.shape(X_train), np.shape(X_val), np.shape(X_test))

#Standard normalize the training data and use the mean and std to normalize the testing data
knn_scaler = StandardScaler().fit(X_train)
X_train = knn_scaler.transform(X_train)
X_test = knn_scaler.transform(X_test)
X_val = knn_scaler.transform(X_val)

# print("Training set ", np.shape(X_train), np.shape(y_train))
# print("Testing set ", np.shape(X_test), np.shape(y_test))


(15031, 5) (3221, 5) (3221, 5)


In [44]:
estimator_KNN = KNeighborsRegressor(algorithm='auto')

parameters_KNN = {
    'n_neighbors': [3, 5, 10],
    'p': [1,2],
    'weights': ['uniform', 'distance'],
    'metric': ['minkowski', 'chebyshev']}

grid_search_KNN = GridSearchCV(
    estimator= estimator_KNN,
    param_grid=parameters_KNN,
    scoring = 'neg_mean_squared_error',#CHANGE THIS
    cv = 5)

parameters = {'kernel':[('rbf')], 'C':[0.01, 0.1, 1, 10, 100, 1000], 'gamma':[0.001, 0.01, 0.1, 1, 10]}

grid_search_KNN.fit(X_train, y_train)
best_model_knn = grid_search_KNN.best_estimator_

print(grid_search_KNN.best_params_)

{'metric': 'minkowski', 'n_neighbors': 3, 'p': 2, 'weights': 'distance'}


In [45]:
y_pred = best_model_knn.predict(X_test)

# Calculate absolute and relative errors for each mass component 
X_test_unnormalized = knn_scaler.inverse_transform(X_test)
mass1i = np.exp(X_test_unnormalized[:, 3])
mass2i = np.exp(X_test_unnormalized[:, 4])

initial_total_masses = mass1i + mass2i # in Msun 
pred_mass1 = y_pred[:,0] * initial_total_masses
pred_mass2 = y_pred[:,1] * initial_total_masses
pred_ejec  = y_pred[:,2] * initial_total_masses

true_mass1 = y_test[:,0] * initial_total_masses
true_mass2 = y_test[:,1] * initial_total_masses
true_ejec  = y_test[:,2] * initial_total_masses

#--Error metric 1: Median Absolute Errors for the respective masses 
median_abs_error_m1 = np.median(np.abs(pred_mass1 - true_mass1))
median_abs_error_m2 = np.median(np.abs(pred_mass2 - true_mass2))
median_abs_error_ejec = np.median(np.abs(pred_ejec - true_ejec))


#--Error metric 2: Relative errors for cases where at least one star survives

median_rel_error_m1 = np.median(np.abs(pred_mass1[true_mass1 != 0.] - true_mass1[true_mass1 != 0. ])/ true_mass1[true_mass1 != 0.])
median_rel_error_m2 = np.median(np.abs(pred_mass2[true_mass2 != 0.] - true_mass2[true_mass2 != 0. ])/ true_mass2[true_mass2 != 0.])
median_rel_error_m_ejec = np.median(np.abs(pred_ejec[true_ejec != 0.] - true_ejec[true_ejec != 0. ])/ true_ejec[true_ejec != 0.])


print(f"Absolute Errors M1 [Msun]: {median_abs_error_m1:.4f}")
print(f"Absolute Errors M2 [Msun]: {median_abs_error_m2:.4f}")
print(f"Relative Errors M1,f  : {median_rel_error_m1:.4f}")
print(f"Relative Errors M2,f: {median_rel_error_m2:.4f}")

Absolute Errors M1 [Msun]: 0.0338
Absolute Errors M2 [Msun]: 0.0004
Relative Errors M1,f  : 0.0066
Relative Errors M2,f: 0.0235


In [91]:
def knn_regression_plotter(X_train, y_train, X_test, y_test, best_model_knn, knn_scaler,
                       feature_idx=(0, 1), fixed_values={}, labels=[] ):
    """
    Plots regression outputs for a PyTorch model using a 2D slice of a higher-dimensional space.
    Produces one panel per output dimension (color gradient).
    
    Parameters:
    - best_model_knn: Trained PyTorch model.
    - X_train, y_train, X_test, y_test: datasets
    - knn_scaler: normalization stats.
    - feature_idx: Tuple (i, j) specifying which two features to plot.
    - fixed_values: {feature_index: value} for fixing other dimensions.
    - labels: list of strings for axis and titles. Expected: [x_label, y_label, ..., etc].
    """
    # Step 1: Normalize fixed values

    fixed_values_norm = {k: (v - knn_scaler.mean_[k]) / knn_scaler.scale_[k] for k, v in fixed_values.items()}

    train_mask = np.all(np.array([np.isclose(X_train[:, k], v, rtol=0.09) for k, v in fixed_values_norm.items()]), axis=0)
    test_mask = np.all(np.array([np.isclose(X_test[:, k], v, rtol=0.09) for k, v in fixed_values_norm.items()]), axis=0)
   
    if (train_mask.sum() == 0 and test_mask.sum() == 0):
        print("No data points!")
        return "No data points!"

    X_train_filtered, y_train_filtered = X_train[train_mask], y_train[train_mask]
    X_test_filtered, y_test_filtered = X_test[test_mask], y_test[test_mask]

    #Convert masses into not logged 
    X_train_filtered_unnorm = knn_scaler.inverse_transform(X_train_filtered)
    X_test_filtered_unnorm = knn_scaler.inverse_transform(X_test_filtered)
    X_train_mass1 = np.exp(X_train_filtered_unnorm[:,3])
    X_train_mass2 = np.exp(X_train_filtered_unnorm[:,4])

    X_test_mass1 = np.exp(X_test_filtered_unnorm[:,3])
    X_test_mass2 = np.exp(X_test_filtered_unnorm[:,4])

    y_train_filtered_unnorm = np.array(y_train_filtered) 
    y_test_filtered_unnorm = np.array(y_test_filtered) 

    # Correct Units 
    y_train_filtered_unnorm[:,0] = y_train_filtered_unnorm[:,0] * (X_train_mass1 + X_train_mass2)
    y_train_filtered_unnorm[:,1] = y_train_filtered_unnorm[:,1] * (X_train_mass1 + X_train_mass2)

    y_test_filtered_unnorm[:,0] = y_test_filtered_unnorm[:,0] * (X_test_mass1 + X_test_mass2)
    y_test_filtered_unnorm[:,1] = y_test_filtered_unnorm[:,1] * (X_test_mass1 + X_test_mass2)
    
    x_min, x_max = X_train_filtered[:, feature_idx[0]].min() - 0.1, X_train_filtered[:, feature_idx[0]].max() + 0.1
    y_min, y_max = X_train_filtered[:, feature_idx[1]].min() - 0.1, X_train_filtered[:, feature_idx[1]].max() + 0.1

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                         np.linspace(y_min, y_max, 500))

    # Step 3: Construct full-dimensional input space for predictions
    X_grid = np.zeros((xx.ravel().shape[0], X_train.shape[1]))
    X_grid[:, feature_idx[0]] = xx.ravel()
    X_grid[:, feature_idx[1]] = yy.ravel()
    
    for k, v in fixed_values.items():
        X_grid[:, k] = v

    #transform the data
    X_grid_scaled = knn_scaler.transform(X_grid)

    #Predict labels for meshgrid
    preds = best_model_knn.predict(X_grid_scaled)
    preds = preds.reshape(xx.shape)

    # Changing the training and testing data to be M1f and M2f 
    y_train_filtered_corrected = np.empty((len(y_train_filtered_unnorm[:,0]), 2))
    y_test_filtered_corrected = np.empty((len(y_test_filtered_unnorm[:,0]), 2))
    
    y_train_filtered_corrected[:,0] = y_train_filtered_unnorm[:, 0]
    y_train_filtered_corrected[:,1] = y_train_filtered_unnorm[:, 1] 

    y_test_filtered_corrected[:,0] = y_test_filtered_unnorm[:, 0]
    y_test_filtered_corrected[:,1] = y_test_filtered_unnorm[:, 1] 

    M1_f = preds[:,0] * (np.exp(fixed_values[3]) + np.exp(fixed_values[4]))
    M2_f = preds[:,1] * (np.exp(fixed_values[3]) + np.exp(fixed_values[4]))

    # Reshape into grid for each output dimension
    M1_f = M1_f.reshape(xx.shape)
    M2_f = M2_f.reshape(xx.shape)

    # Step 5: Plot two panels
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    vmin = min(y_train_filtered_corrected[:,0].min(), y_train_filtered_corrected[:,1].min(),
           y_test_filtered_corrected[:,0].min(), y_test_filtered_corrected[:,1].min(),
           M1_f.min(), M2_f.min())
    vmax = max(y_train_filtered_corrected[:,0].max(), y_train_filtered_corrected[:,1].max(),
           y_test_filtered_corrected[:,0].max(), y_test_filtered_corrected[:,1].max(),
           M1_f.max(), M2_f.max())
    
    cmap = plt.cm.coolwarm
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for i, (Z, ax, title) in enumerate(zip([M1_f, M2_f], axes, ['Star 1 Final Mass', 'Star 2 Final Mass'])):
        im = ax.contourf(xx, yy, Z, levels = 100, cmap=cmap, norm = norm)
        scatter1 = ax.scatter(X_train_filtered[:, feature_idx[0]],
                              X_train_filtered[:, feature_idx[1]],
                              c=y_train_filtered_corrected[:, i], cmap=cmap, norm = norm, edgecolor="k", marker="o", label="Train")
        scatter2 = ax.scatter(X_test_filtered[:, feature_idx[0]],
                              X_test_filtered[:, feature_idx[1]],
                              c=y_test_filtered_corrected[:, i], cmap=cmap, norm = norm, marker="^", label="Test")

        ax.set_xlabel(fr"{labels[0]}", fontsize=14)
        ax.set_ylabel(fr"{labels[1]}", fontsize=14)
        ax.tick_params(axis='both', which='major', labelsize=12)
        ax.set_title(title, fontsize=15)
    
   
    # After plotting the contours and scatters
    plt.tight_layout(rect=[0,0,0.9,1])  # leave 10% space on the right for the colorbar

    # Create the colorbar on a dedicated axis outside the panels
    cbar_ax = fig.add_axes([0.9999, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                    cax=cbar_ax, orientation='vertical', shrink=0.8)
    cbar.set_label('Final Mass', fontsize=14)

    fig.suptitle(fr"$\mathrm{{M_1}} = {labels[2]}\ M_\odot,\ \mathrm{{M_2}} = {labels[3]}\ M_\odot,\ \mathrm{{Time}} = {round(10**(float(labels[4])),3)}\ \mathrm{{Gyr}}$", fontsize = 17)

    plt.tight_layout()
    return fig




In [92]:
# unique_rows = unique_rows.T
import matplotlib.colors as mcolors
unique_rows = np.array([[1.0,1.0]])
for unique in unique_rows:
    Mass1 = np.log(unique[0])
    Mass2 = np.log(unique[1])
    # labels = ['log10(b[RSUN])', 'log10(v_inf[km/s])', str(Mass1), str(round(Mass2,2))]
    labels = ['log10(rp/(R1+R2))', 'log10(v_inf[km/s])', str(Mass1), str(round(Mass2,2))]
    fig = knn_regression_plotter(X_train, y_train, X_test ,y_test, best_model_knn, knn_scaler,feature_idx=(0, 1), fixed_values={3: Mass1, 4: Mass2}, labels = labels)
    plt.show()

ValueError: cannot reshape array of size 750000 into shape (500,500)